In [1]:
import os
import numpy as np
import h5py

In [2]:
root_dir = "/system/user/publicdata/gyrokinetics"
folder = "cyclone4"
dir_in = os.path.join(root_dir, folder)
dir_out = "/system/user/publicdata/gyrokinetics_preprocessed"
if not os.path.exists(dir_out):
    os.makedirs(dir_out)

In [58]:
# create list of Ks
ks = ["K" + str(i).zfill(2) for i in range(1, 51)]

In [60]:
# get timestamps
ts = []
for k in ks:
    # load corresponding timestep
    with open(os.path.join(dir_in, k + ".dat"), "r") as file:
        for line in file:
            line_split = line.split("=")
            if line_split[0].strip() == "TIME":
                time = float(line_split[1].strip().strip(",").strip())
                ts.append(time)
timesteps = np.array(ts)

# read helper vars
time = np.loadtxt(os.path.join(dir_in, "time.dat"))
sgrid = np.loadtxt(os.path.join(dir_in, "sgrid"))
xphi = np.loadtxt(os.path.join(dir_in, "xphi"))
yphi = np.loadtxt(os.path.join(dir_in, "yphi"))
fluxes = np.loadtxt(os.path.join(dir_in, "fluxes.dat"))
kyspec = np.loadtxt(os.path.join(dir_in, "kyspec"))
krho = np.loadtxt(os.path.join(dir_in, "krho"))
vpgr = np.loadtxt(os.path.join(dir_in, "vpgr.dat"))
vperp = np.loadtxt(os.path.join(dir_in, "vperp.dat"))
# number of parallel direction grid points
ns = sgrid.shape[1] if len(sgrid.shape) > 1 else sgrid.shape[0]
# number of x, y grid points (in real space)
nx, ny = xphi.shape[1], xphi.shape[0]
# number of modes in x and y direction
nkx, nky = krho.shape[1], krho.shape[0]
# get velocity space resolutions
nvpar, nmu = vpgr.shape[1], vpgr.shape[0]

# create h5 file with timestamps and field data
h5_filename = os.path.join(dir_out, folder + ".h5")
with h5py.File(h5_filename, "w") as file:
    # group for metadata (e.g. timesteps)
    metadata_group = file.create_group("metadata")
    metadata_group.create_dataset("timesteps", data=timesteps)

    # group for our 6D field data
    data_group = file.create_group("data")
    for idx, k in enumerate(ks):
        # Load the full distribution function data
        with open(os.path.join(dir_in, k), "rb") as fid:
            ff = np.fromfile(fid, dtype=np.float64)

        # Reshape the distribution function
        f = np.reshape(ff, (2, nvpar, nmu, ns, nkx, nky), order="F").astype("float32")

        # Add the reshaped data as a dataset to the "data" group
        dataset_name = "timestep_" + str(idx).zfill(2)
        data_group.create_dataset(dataset_name, data=f)

        print(f"Added dataset '{dataset_name}' to HDF5 file.")

Added dataset 'timestep_00' to HDF5 file.
Added dataset 'timestep_01' to HDF5 file.
Added dataset 'timestep_02' to HDF5 file.
Added dataset 'timestep_03' to HDF5 file.
Added dataset 'timestep_04' to HDF5 file.
Added dataset 'timestep_05' to HDF5 file.
Added dataset 'timestep_06' to HDF5 file.
Added dataset 'timestep_07' to HDF5 file.
Added dataset 'timestep_08' to HDF5 file.
Added dataset 'timestep_09' to HDF5 file.
Added dataset 'timestep_10' to HDF5 file.
Added dataset 'timestep_11' to HDF5 file.
Added dataset 'timestep_12' to HDF5 file.
Added dataset 'timestep_13' to HDF5 file.
Added dataset 'timestep_14' to HDF5 file.
Added dataset 'timestep_15' to HDF5 file.
Added dataset 'timestep_16' to HDF5 file.
Added dataset 'timestep_17' to HDF5 file.
Added dataset 'timestep_18' to HDF5 file.
Added dataset 'timestep_19' to HDF5 file.
Added dataset 'timestep_20' to HDF5 file.
Added dataset 'timestep_21' to HDF5 file.
Added dataset 'timestep_22' to HDF5 file.
Added dataset 'timestep_23' to HDF

In [61]:
# read in the structure and example field of the created h5 file
with h5py.File(h5_filename, "r") as f:
    # Read the 'metadata/timesteps' dataset
    timesteps = f["metadata/timesteps"][:]
    print("Timesteps:", timesteps)

    # Read the 'data/timestep_0' dataset
    timestep_0 = f["data/timestep_00"][:]
    print("Shape of timestep_00:", timestep_0.shape)


Timesteps: [  5.          10.          14.69028539  18.04342226  21.97145444
  26.12001126  30.05082011  34.25605765  38.30417678  42.72323897
  46.90625505  51.33567014  55.64891717  60.24671935  64.6930198
  69.31442234  74.16218674  78.79486829  83.42996536  88.32654589
  92.59371514  96.50930751 100.83579437 105.66562344 110.39540365
 115.18905111 119.72558046 123.97167886 128.48791957 132.76599482
 137.35942298 141.61653482 146.26823364 151.04355177 155.79254785
 160.19116154 165.01472419 169.73205348 174.47434844 179.31991695
 184.26068711 188.83986478 193.51736551 197.78700782 201.41570539
 205.71878393 210.1754861  214.3448279  218.92316443 223.48979821]
Shape of timestep_00: (2, 32, 8, 16, 167, 21)
